<div style="text-align: right;">
Regina Tamayo León<br>
Valeria Estefanía Milke Loera<br>
Elisa Aguirre Arias
</div>

# **P03-: Análisis Predictivo de la Calidad del Vino Tinto mediante Modelos de Ensamble y Optimización de Hiperparámetros**

### 1. Objetivos

**1.1 Objetivo General**: Desarrollar y comparar modelos de aprendizaje automático (Random Forest y XGBoost) para clasificar la calidad del vino tinto basada en sus propiedades fisicoquímicas, optimizando su rendimiento mediante la búsqueda de hiperparámetros y validación cruzada.

**1.2 Objetivos específicos**: 
- Realizar un análisis exploratorio de datos (EDA) para entender la distribución y características del dataset WineQT.
- Implementar un pipeline de preprocesamiento que asegure la limpieza y normalización de los datos.
- Aplicar técnicas de optimización (Random Search) para encontrar los mejores hiperparámetros de cada modelo.
- Evaluar el desempeño de los modelos utilizando métricas de precisión (Accuracy) mediante $k$-folds cross-validation para garantizar la estabilidad de los resultados.
- Interpretar los resultados obtenidos para concluir sobre la viabilidad del modelo en un entorno de producción vinícola.
 __________________

### 2. Marco Teórico
**Regresión Lineal y sus 6 Problemas Potenciales (ISLP)**

La regresión lineal es un método estadístico para modelar la relación entre una variable dependiente y una o más independientes. Según el libro An Introduction to Statistical Learning (ISLP), existen 6 problemas principales que pueden invalidar el modelo:
1. No linealidad de la relación entre respuesta y predictores: Si el fenómeno no es lineal, el modelo tendrá un sesgo alto.
2. Correlación de los términos de error: Viola la independencia de los errores (común en series de tiempo).
3. Varianza no constante de los términos de error (Heterocedasticidad): Los errores cambian su dispersión según el valor de los predictores.
4. Valores atípicos (Outliers): Observaciones con valores de $y$ inusuales que pueden distorsionar la estimación.
5. Puntos de alto apalancamiento (High-leverage points): Observaciones con valores de $x$ inusuales que afectan desproporcionadamente la pendiente.
6. Colinealidad: Cuando dos o más predictores están altamente correlacionados entre sí, dificultando la interpretación de los coeficientes.

**Árboles de Decisión**

*Árbol para Regresión:* Divide el espacio de predictores en regiones y asigna el valor promedio de la respuesta en esa región.

*Árbol para Clasificación:* Similar al de regresión, pero predice la clase más común (moda) en la región mediante criterios como la impureza de Gini o Entropía.

**Bootstrap**:

Método de remuestreo con reemplazo que permite estimar la variabilidad de un estadístico.

**Ensambles de modelos**: 

Combinación de múltiples modelos base para obtener una predicción más robusta.

**Bagging** (Bootstrap Aggregating): 

Entrena múltiples modelos en paralelo sobre muestras bootstrap y promedia sus resultados (ej. Random Forest).

**Boosting**: 

Entrena modelos de forma secuencial, donde cada nuevo modelo intenta corregir los errores del anterior (ej. XGBoost).

**Temas relacionados con el vino**

**Quimiometría:** 

Aplicación de métodos matemáticos y estadísticos a datos de origen químico. En este proyecto, usamos la quimiometría para traducir niveles de sulfatos y azúcares en una nota de cata.

**Análisis Sensorial vs. Fisicoquímico:** 

La calidad es una variable subjetiva asignada por humanos, pero el modelo busca los "proxys" químicos (objetivos) que explican esa subjetividad.

**Desbalance de Clases:** 

En el vino, hay muchos vinos "promedio" (calidad 5 y 6) y muy pocos "excelentes" (calidad 8). Esto requiere técnicas de validación estratificada.
__________________

### 3. Análisis del dataset 

**¿De dónde viene?:**

El dataset WineQT (Wine Quality Dataset) es una versión filtrada del famoso estudio de la Universidad de Minho (Portugal). Contiene datos de la variante tinta del vino "Vinho Verde".

**¿Qué contiene? (Variables Clave):**

Acidez Volátil: El exceso puede dar sabor a vinagre.

Ácido Cítrico: Aporta frescura.

Azúcar Residual: Influye en el cuerpo y dulzor.

Cloruros: Cantidad de sal; niveles altos son indeseables.

Alcohol: Generalmente el predictor más fuerte de calidad percibida.

pH: Describe la acidez o alcalinidad (escala logarítmica).

**¿Qué información dan las muestras?:**

Las 1143 muestras muestran un sesgo hacia calidades medias. Observamos que los vinos de mayor calidad tienden a tener menores niveles de acidez volátil y mayores niveles de alcohol y sulfatos (que actúan como conservantes).

**¿Qué se quiere analizar?:**

Se busca determinar si es posible sustituir o apoyar la cata humana con un análisis químico automatizado. Queremos identificar el "umbral químico" que separa un vino mediocre de uno premium.

**¿Qué resultado se podría encontrar al ajustar un modelo?:**

Se espera que el Random Forest maneje mejor las relaciones no lineales entre el pH y la densidad.

Se anticipa que el XGBoost logre una mayor precisión al enfocarse en las clases difíciles (vinos de calidad 3 u 8), aunque podría requerir más ajuste para evitar el overfitting.
__________________

### 4.Modelo propuesto y Pipeline

In [2]:
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 12.3 MB/s eta 0:00:00a 0:00:01


In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline

df = pd.read_csv('WineQT.csv')
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,2
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,3
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,4


In [10]:
X = df.drop(['quality', 'Id'], axis=1)
y = df['quality']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

Pipeline:Escalamiento + Clasificador

In [13]:
rf_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(random_state=42))
])

# hiperparámetros
rf_param_grid = {
    'rf__n_estimators': [100, 200, 300, 500],
    'rf__max_depth': [None, 10, 20, 30],
    'rf__min_samples_split': [2, 5, 10],
    'rf__criterion': ['gini', 'entropy']
}

Optimización con búsqueda aleatoria

In [24]:
rf_search = RandomizedSearchCV(rf_pipe, rf_param_grid, n_iter=20, cv=5, 
                               scoring='accuracy', random_state=42, n_jobs=-1)
rf_search.fit(X_train, y_train)

RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                             ('rf',
                                              RandomForestClassifier(random_state=42))]),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'rf__criterion': ['gini', 'entropy'],
                                        'rf__max_depth': [None, 10, 20, 30],
                                        'rf__min_samples_split': [2, 5, 10],
                                        'rf__n_estimators': [100, 200, 300,
                                                             500]},
                   random_state=42, scoring='accuracy')

Pipeline:Escalamiento + Clasificador

In [25]:
xgb_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', XGBClassifier(random_state=42, eval_metric='mlogloss'))
])

# hiperparámetros
xgb_param_grid = {
    'xgb__n_estimators': [100, 200, 300],
    'xgb__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'xgb__max_depth': [3, 5, 7, 9],
    'xgb__subsample': [0.7, 0.8, 0.9]
}


Optimización

In [26]:
xgb_search = RandomizedSearchCV(xgb_pipe, xgb_param_grid, n_iter=20, cv=5, 
                                scoring='accuracy', random_state=42, n_jobs=-1)
xgb_search.fit(X_train, y_train)

RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                             ('xgb',
                                              XGBClassifier(base_score=None,
                                                            booster=None,
                                                            callbacks=None,
                                                            colsample_bylevel=None,
                                                            colsample_bynode=None,
                                                            colsample_bytree=None,
                                                            device=None,
                                                            early_stopping_rounds=None,
                                                            enable_categorical=False,
                                                            eval_metric='mlogloss',
                                                            feature_types=None,
                                                            feature_weights=None,
                                                            gamma=None,
                                                            gr...
                                                            min_child_weight=None,
                                                            missing=nan,
                                                            monotone_constraints=None,
                                                            multi_strategy=None,
                                                            n_estimators=None,
                                                            n_jobs=None,
                                                            num_parallel_tree=None, ...))]),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'xgb__learning_rate': [0.01, 0.05, 0.1,
                                                               0.2],
                                        'xgb__max_depth': [3, 5, 7, 9],
                                        'xgb__n_estimators': [100, 200, 300],
                                        'xgb__subsample': [0.7, 0.8, 0.9]},
                   random_state=42, scoring='accuracy')

In [27]:
# Definición de la validación cruzada estratificada
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

Comparación del métrico (Accuracy)

In [28]:
rf_final_scores = cross_val_score(rf_search.best_estimator_, X_train, y_train, cv=cv_strategy)
xgb_final_scores = cross_val_score(xgb_search.best_estimator_, X_train, y_train, cv=cv_strategy)

print(f"Random Forest - Accuracy Media: {np.mean(rf_final_scores):.4f} (+/- {np.std(rf_final_scores):.4f})")
print(f"XGBoost       - Accuracy Media: {np.mean(xgb_final_scores):.4f} (+/- {np.std(xgb_final_scores):.4f})")

Random Forest - Accuracy Media: 0.6598 (+/- 0.0506)
XGBoost       - Accuracy Media: 0.6324 (+/- 0.0328)


In [19]:
# Identificar al ganador comparando las medias
if np.mean(xgb_final_scores) > np.mean(rf_final_scores):
    mejor_modelo = xgb_search.best_estimator_
    nombre_modelo = "XGBoost"
else:
    mejor_modelo = rf_search.best_estimator_
    nombre_modelo = "Random Forest"

Calcular el Accuracy Final con el X_test (datos nunca vistos)

In [23]:
final_pred = mejor_modelo.predict(X_test)
from sklearn.metrics import accuracy_score, classification_report

accuracy_test = accuracy_score(y_test, final_pred)

print(f"-EVALUACIÓN FINAL DEL PROYECTO-")
print(f"El ganador es: {nombre_modelo}")
print(f"Accuracy en el Test Set: {accuracy_test:.4f}")

# Ver en qué categorías acierta más
print("Reporte de Clasificación:")
print(classification_report(y_test, final_pred))

-EVALUACIÓN FINAL DEL PROYECTO-
El ganador es: Random Forest
Accuracy en el Test Set: 0.6900
Reporte de Clasificación:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.00      0.00      0.00         7
           2       0.72      0.81      0.77        97
           3       0.65      0.68      0.67        92
           4       0.70      0.55      0.62        29
           5       0.00      0.00      0.00         3

    accuracy                           0.69       229
   macro avg       0.34      0.34      0.34       229
weighted avg       0.66      0.69      0.67       229



/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


**Conclusiones**

Tras la optimización de hiperparámetros, el modelo Random Forest demostró una mayor capacidad predictiva en comparación con XGBoost para este dataset específico. Esto sugiere que, para un conjunto de datos de este tamaño (1,143 registros), los bosques aleatorios pueden ser más robustos frente al ruido que los modelos de 'boosting' intensivo.

La baja desviación estándar obtenida mediante la validación cruzada ($k$-folds) confirma que los modelos son estables y no dependen de una partición específica de los datos. El hecho de que el accuracy en el set de prueba ($X\_test$) sea cercano a la media de la validación cruzada indica que el modelo tiene una excelente capacidad de generalización y no presenta overfitting.

El modelo presenta un desempeño notable y consistente en la clasificación de las calidades de vino más representadas en el dataset (etiquetas 2 y 3, que corresponden a las calidades 5 y 6), alcanzando F1-scores de 0.77 y 0.67. La cercanía entre los resultados de precisión y sensibilidad (recall) en estas categorías indica que el modelo ha logrado aprender patrones robustos para identificar vinos de calidad estándar.

Sin embargo, el reporte evidencia una limitación crítica en las clases minoritarias (etiquetas 0, 1 y 5). Debido a que estas categorías cuentan con un soporte extremadamente bajo (menos de 10 muestras), el modelo es incapaz de identificarlas, resultando en métricas de 0.00.

**Referencias APA**

Yasser, H. (2022). Wine Quality Dataset [Conjunto de datos]. Kaggle. https://www.kaggle.com/datasets/yasserh/wine-quality-dataset

Breiman, L. (2001). Random forests. Machine Learning, 45(1), 5-32. https://doi.org/10.1023/A:1010933404324

Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining, 785-794. https://doi.org/10.1145/2939672.2939785

